# 🧠 Long Horizon Memory - Comprehensive Benchmarking Suite

**Run this on Kaggle with GPU (Blackwell 5090 recommended)**

This notebook provides:
- Multiple model comparisons
- Comprehensive metrics
- Visualization and analysis
- Export results for submission

## 📦 Setup & Installation

In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes
!pip install -q matplotlib seaborn pandas numpy
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Upload your entire rl/ folder to Kaggle first, then adjust this path
import sys
sys.path.insert(0, '/kaggle/input/long-horizon-memory')  # Adjust to your Kaggle dataset path

import json
import time
import os
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 🤖 Model Loading (Qwen 32B with 4-bit Quantization)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit quantization config for 32B model on 24GB VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading Qwen2.5-32B-Instruct with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-32B-Instruct",
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-32B-Instruct",
    trust_remote_code=True
)

print(f"✓ Model loaded on: {model.device}")
print(f"✓ Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 📋 Load Environment & Episodes

In [ ]:
# Load episodes
with open('/kaggle/input/long-horizon-memory/server/episodes.json', 'r') as f:
    episodes_data = json.load(f)

print(f"Loaded {len(episodes_data)} episodes")
print(f"Difficulties: {pd.Series([ep['difficulty'] for ep in episodes_data]).value_counts().to_dict()}")

In [ ]:
# Import environment components
from models import LongHorizonMemoryAction, LongHorizonMemoryObservation

# System prompt (copy from inference.py)
SYSTEM_PROMPT = """You are an expert memory management system for long-horizon tasks. Your goal is to maintain a high-quality memory buffer by keeping relevant information and discarding noise.

SCORING FORMULA (aim for ≥0.7 for success):
task_score = 0.6×recall + 0.4×precision - 0.25×incorrect_rate - 0.15×overflow_rate

KEY INSIGHTS:
- Recall (60% weight) is MORE important than precision (40%)
- Memory capacity: 8 slots maximum
- Prioritize capturing ALL relevant info, but avoid keeping irrelevant distractors
- Better to keep relevant items even if some noise, than miss important info

RELEVANCE INDICATORS (keep these):
- Technical problems, bugs, requirements, constraints
- User goals, pain points, specific needs
- System design decisions, architecture details
- Performance issues, metrics, monitoring needs
- Domain-specific technical details

IRRELEVANCE INDICATORS (discard these):
- Personal hobbies (hobbies, sports, cooking, photography, music)
- Shopping and purchases (buying items, products)
- Lifestyle and daily activities (weekend plans, food, exercise)
- Entertainment (movies, shows, games, books unrelated to task)
- Random observations with no technical connection

STRATEGY:
1. ADD: If message contains technical content relevant to the domain
2. REMOVE: If memory contains noise AND you need space for better content
3. NOOP: If current message is noise and memory is already optimal

CRITICAL: You must output ONLY valid JSON with this exact format:
{"operation": "add"}
OR
{"operation": "remove", "remove_index": 0}
OR
{"operation": "noop"}

Think carefully about each decision based on the scoring formula."""

## 🎯 Agent Implementation

In [ ]:
def generate_response(messages: List[Dict], temperature: float = 0.1, max_tokens: int = 150) -> str:
    """Generate response from local Qwen model."""
    
    # Format messages for Qwen chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()


def parse_action(content: str) -> Dict:
    """Parse JSON action from LLM output."""
    normalized = content.strip()
    
    # Remove markdown code blocks
    if normalized.startswith("```"):
        normalized = normalized.strip("`")
        normalized = normalized.replace("json", "", 1).strip()
    
    try:
        return json.loads(normalized)
    except Exception as e:
        print(f"Parse error: {e}")
        print(f"Content: {content[:200]}")
        return {"operation": "noop"}  # Safe fallback


def choose_action_llm(
    memory: List[str],
    memory_count: int,
    new_message: str,
    domain: str,
    task_name: str,
    metadata: Dict
) -> Dict:
    """Choose action using LLM with chain-of-thought prompting."""
    
    # Build context
    memory_summary = (
        f"Currently storing {memory_count}/8 items in memory.\n"
        f"Current memory contents:\n"
        + "\n".join(f"  [{i}] {msg}" for i, msg in enumerate(memory))
        if memory
        else "Memory is empty (0/8 items)."
    )
    
    current_score = metadata.get("task_score", 0.0)
    correct_count = metadata.get("correct_in_memory", 0)
    incorrect_count = metadata.get("incorrect_in_memory", 0)
    
    user_prompt = f"""TASK DIFFICULTY: {task_name}
DOMAIN: {domain}

CURRENT STATE:
{memory_summary}

PERFORMANCE METRICS:
- Current task score: {current_score:.2f} (need ≥0.70 for success)
- Correct items in memory: {correct_count}
- Incorrect items in memory: {incorrect_count}
- Recall: {correct_count}/{metadata.get('memory_capacity', 8)} slots used

NEW INCOMING MESSAGE:
"{new_message}"

DECISION REQUIRED:
Analyze if the new message is relevant to the {domain} domain task.
Consider current memory state and whether you need to make room or keep current optimal state.
Output your decision as JSON only."""
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]
    
    response = generate_response(messages)
    return parse_action(response)

## 🏃 Simple Episode Runner (Manual Simulation)

In [ ]:
@dataclass
class EpisodeResult:
    episode_id: int
    difficulty: str
    domain: str
    success: bool
    final_score: float
    steps: int
    avg_reward: float
    rewards: List[float]
    correct_final: int
    incorrect_final: int
    decisions: List[Dict]
    inference_time: float


def run_episode_manual(episode: Dict) -> EpisodeResult:
    """Run a single episode with manual scoring."""
    
    start_time = time.time()
    
    episode_id = episode['episode_id']
    difficulty = episode['difficulty']
    domain = episode['conversation_domain']
    messages = episode['string_relevant_messages']
    
    # State
    memory = []
    rewards = []
    decisions = []
    
    # Ground truth
    relevant_messages = [msg['text'] for msg in messages if msg['isRelevant']]
    
    print(f"\n[EPISODE {episode_id}] {difficulty} - {domain}")
    
    for step, msg_obj in enumerate(messages, 1):
        msg_text = msg_obj['text']
        is_relevant = msg_obj['isRelevant']
        
        # Calculate current metrics
        correct_in_memory = sum(1 for m in memory if m in relevant_messages)
        incorrect_in_memory = len(memory) - correct_in_memory
        
        recall = correct_in_memory / len(relevant_messages) if relevant_messages else 0
        precision = correct_in_memory / len(memory) if memory else 0
        incorrect_rate = incorrect_in_memory / 8
        overflow_rate = max(0, len(memory) - len(relevant_messages)) / 8
        
        task_score = 0.6 * recall + 0.4 * precision - 0.25 * incorrect_rate - 0.15 * overflow_rate
        task_score = max(0.0, min(1.0, task_score))
        
        metadata = {
            "task_score": task_score,
            "correct_in_memory": correct_in_memory,
            "incorrect_in_memory": incorrect_in_memory,
            "memory_capacity": 8
        }
        
        # Get action from LLM
        action = choose_action_llm(
            memory=memory,
            memory_count=len(memory),
            new_message=msg_text,
            domain=domain,
            task_name=difficulty,
            metadata=metadata
        )
        
        # Execute action
        operation = action.get("operation", "noop")
        
        if operation == "add":
            if len(memory) < 8:
                memory.append(msg_text)
        elif operation == "remove":
            remove_idx = action.get("remove_index", 0)
            if 0 <= remove_idx < len(memory):
                memory.pop(remove_idx)
        # noop: do nothing
        
        # Recalculate reward after action
        correct_in_memory = sum(1 for m in memory if m in relevant_messages)
        incorrect_in_memory = len(memory) - correct_in_memory
        recall = correct_in_memory / len(relevant_messages) if relevant_messages else 0
        precision = correct_in_memory / len(memory) if memory else 0
        incorrect_rate = incorrect_in_memory / 8
        overflow_rate = max(0, len(memory) - len(relevant_messages)) / 8
        
        task_score = 0.6 * recall + 0.4 * precision - 0.25 * incorrect_rate - 0.15 * overflow_rate
        task_score = max(0.0, min(1.0, task_score))
        
        reward = task_score
        rewards.append(reward)
        
        decisions.append({
            "step": step,
            "message": msg_text,
            "is_relevant": is_relevant,
            "action": operation,
            "reward": reward,
            "score": task_score,
            "correct": correct_in_memory,
            "incorrect": incorrect_in_memory
        })
        
        print(f"  [{step}] {operation:6s} | reward={reward:.3f} | score={task_score:.3f} | correct={correct_in_memory} incorrect={incorrect_in_memory}")
    
    # Final metrics
    final_score = rewards[-1] if rewards else 0.0
    success = final_score >= 0.70
    avg_reward = np.mean(rewards) if rewards else 0.0
    
    correct_final = sum(1 for m in memory if m in relevant_messages)
    incorrect_final = len(memory) - correct_final
    
    inference_time = time.time() - start_time
    
    print(f"  [RESULT] Success={success} | Final Score={final_score:.3f} | Time={inference_time:.1f}s")
    
    return EpisodeResult(
        episode_id=episode_id,
        difficulty=difficulty,
        domain=domain,
        success=success,
        final_score=final_score,
        steps=len(messages),
        avg_reward=avg_reward,
        rewards=rewards,
        correct_final=correct_final,
        incorrect_final=incorrect_final,
        decisions=decisions,
        inference_time=inference_time
    )

## 🚀 Run Full Benchmark

In [ ]:
# Run all 24 episodes
results = []

for episode in episodes_data:
    result = run_episode_manual(episode)
    results.append(result)

print(f"\n{'='*80}")
print(f"BENCHMARK COMPLETE: {len(results)} episodes")
print(f"{'='*80}")

## 📊 Analysis & Visualization

In [ ]:
# Convert to DataFrame
df = pd.DataFrame([{
    'episode_id': r.episode_id,
    'difficulty': r.difficulty,
    'domain': r.domain,
    'success': r.success,
    'final_score': r.final_score,
    'steps': r.steps,
    'avg_reward': r.avg_reward,
    'correct_final': r.correct_final,
    'incorrect_final': r.incorrect_final,
    'inference_time': r.inference_time
} for r in results])

display(df)

In [ ]:
# Summary Statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

total_episodes = len(df)
total_success = df['success'].sum()
success_rate = (total_success / total_episodes) * 100

print(f"\nTotal Episodes: {total_episodes}")
print(f"Successful: {total_success} ({success_rate:.1f}%)")
print(f"Failed: {total_episodes - total_success} ({100 - success_rate:.1f}%)")
print(f"\nAverage Final Score: {df['final_score'].mean():.3f}")
print(f"Median Final Score: {df['final_score'].median():.3f}")
print(f"Std Final Score: {df['final_score'].std():.3f}")
print(f"\nAverage Inference Time: {df['inference_time'].mean():.2f}s")
print(f"Total Inference Time: {df['inference_time'].sum():.2f}s")

print("\n" + "-"*80)
print("BY DIFFICULTY")
print("-"*80)
difficulty_stats = df.groupby('difficulty').agg({
    'success': ['count', 'sum', 'mean'],
    'final_score': ['mean', 'std']
})
display(difficulty_stats)

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Success Rate by Difficulty
difficulty_success = df.groupby('difficulty')['success'].agg(['sum', 'count'])
difficulty_success['rate'] = difficulty_success['sum'] / difficulty_success['count'] * 100
difficulty_success['rate'].plot(kind='bar', ax=axes[0, 0], color=['green', 'orange', 'red'])
axes[0, 0].set_title('Success Rate by Difficulty', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Success Rate (%)')
axes[0, 0].set_xlabel('Difficulty')
axes[0, 0].axhline(y=70, color='black', linestyle='--', label='Target (70%)')
axes[0, 0].legend()

# 2. Score Distribution
axes[0, 1].hist(df['final_score'], bins=20, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=0.70, color='red', linestyle='--', linewidth=2, label='Success Threshold')
axes[0, 1].set_title('Final Score Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Final Score')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# 3. Score by Episode
colors = df['success'].map({True: 'green', False: 'red'})
axes[1, 0].scatter(df['episode_id'], df['final_score'], c=colors, s=100, alpha=0.6)
axes[1, 0].axhline(y=0.70, color='black', linestyle='--', label='Success Threshold')
axes[1, 0].set_title('Score by Episode', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Episode ID')
axes[1, 0].set_ylabel('Final Score')
axes[1, 0].legend()

# 4. Inference Time by Difficulty
df.boxplot(column='inference_time', by='difficulty', ax=axes[1, 1])
axes[1, 1].set_title('Inference Time by Difficulty', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Difficulty')
axes[1, 1].set_ylabel('Inference Time (s)')
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.savefig('benchmark_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved visualization to benchmark_results.png")

## 💾 Export Results

In [ ]:
# Export CSV
df.to_csv('benchmark_results.csv', index=False)
print("✓ Saved benchmark_results.csv")

# Export detailed JSON
detailed_results = {
    'summary': {
        'total_episodes': total_episodes,
        'successful': int(total_success),
        'success_rate': float(success_rate),
        'avg_final_score': float(df['final_score'].mean()),
        'median_final_score': float(df['final_score'].median()),
        'total_inference_time': float(df['inference_time'].sum())
    },
    'episodes': [{
        'episode_id': r.episode_id,
        'difficulty': r.difficulty,
        'domain': r.domain,
        'success': r.success,
        'final_score': r.final_score,
        'steps': r.steps,
        'avg_reward': r.avg_reward,
        'correct_final': r.correct_final,
        'incorrect_final': r.incorrect_final,
        'inference_time': r.inference_time,
        'rewards': r.rewards,
        'decisions': r.decisions
    } for r in results]
}

with open('benchmark_detailed.json', 'w') as f:
    json.dump(detailed_results, f, indent=2)

print("✓ Saved benchmark_detailed.json")
print("\n🎉 Benchmarking complete! Download all files for submission.")